# KL Divergence Analysis: Model vs GroupUniformKnownBayes

This notebook demonstrates how to compute and visualize KL divergence between a trained model's predictions and the Bayesian optimal predictor (GroupUniformKnownBayes).

## Overview

The analysis:
1. Loads a trained model and sampler based on experiment name
2. Samples sequences from major and minor tasks
3. Computes model predictions at padded token positions
4. Compares with GroupUniformKnownBayes predictions on non-padded sequences
5. Plots KL divergence vs position, separated by major/minor tasks

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt
import numpy as np

# Add parent directory to path if needed
sys.path.append('..')

from src.icl.utils.kl_divergence_analysis import (
    compute_kl_divergence_vs_bayes,
    plot_kl_divergence,
    analyze_kl_divergence,
)

## Configuration

Set your experiment name and parameters here.

In [ ]:
# Experiment configuration
EXP_NAME = "your_experiment_name"  # Replace with your experiment name
N_MINOR = 64  # Number of minority tasks to sample
BATCH_SIZE = 64  # Number of sequences per task
STEP = None  # Training step (None = final checkpoint)
P_COMMON = 0.9  # Prior probability for major tasks
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")
print(f"Experiment: {EXP_NAME}")

## Quick Analysis

Use the `analyze_kl_divergence` function for a complete analysis with automatic plotting.

In [ ]:
results = analyze_kl_divergence(
    exp_name=EXP_NAME,
    n_minor=N_MINOR,
    batch_size=BATCH_SIZE,
    step=STEP,
    p_common=P_COMMON,
    device=DEVICE,
    save_path=None,  # Set to a path to save the plot
    show=True,
)

## Detailed Analysis

For more control, you can compute and plot separately.

In [ ]:
# Compute KL divergence
results = compute_kl_divergence_vs_bayes(
    exp_name=EXP_NAME,
    n_minor=N_MINOR,
    batch_size=BATCH_SIZE,
    step=STEP,
    p_common=P_COMMON,
    device=DEVICE,
)

print(f"\nResults computed!")
print(f"Major tasks: {results['n_major_tasks']}")
print(f"Minor tasks: {results['n_minor_tasks']}")
print(f"Positions: {len(results['positions'])}")

## Visualization

Plot the KL divergence with custom styling.

In [ ]:
# Create custom plot
fig, ax = plot_kl_divergence(
    results=results,
    title=f'KL Divergence Analysis: {EXP_NAME}',
    save_path=None,
    show=True,
    figsize=(12, 6),
)

## Additional Analysis

Explore the results in more detail.

In [ ]:
# Analyze specific positions
early_positions = slice(0, 10)
late_positions = slice(-10, None)

print("=== Early Positions (0-10) ===")
print(f"Major tasks - Mean KL: {results['kl_major_mean'][early_positions].mean():.4f}")
print(f"Minor tasks - Mean KL: {results['kl_minor_mean'][early_positions].mean():.4f}")

print("\n=== Late Positions (last 10) ===")
print(f"Major tasks - Mean KL: {results['kl_major_mean'][late_positions].mean():.4f}")
print(f"Minor tasks - Mean KL: {results['kl_minor_mean'][late_positions].mean():.4f}")

# Compute improvement ratio
major_improvement = results['kl_major_mean'][0] / results['kl_major_mean'][-1]
minor_improvement = results['kl_minor_mean'][0] / results['kl_minor_mean'][-1]

print("\n=== Improvement (First / Last) ===")
print(f"Major tasks: {major_improvement:.2f}x")
print(f"Minor tasks: {minor_improvement:.2f}x")

## Per-Task Analysis

Examine KL divergence for individual tasks.

In [ ]:
# Plot individual major tasks
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Major tasks
for i in range(min(5, results['n_major_tasks'])):
    axes[0].plot(results['positions'], results['kl_major'][i], 
                label=f'Task {i}', alpha=0.7)
axes[0].set_xlabel('Position')
axes[0].set_ylabel('KL Divergence')
axes[0].set_title('Major Tasks (first 5)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Minor tasks
for i in range(min(5, results['n_minor_tasks'])):
    axes[1].plot(results['positions'], results['kl_minor'][i], 
                label=f'Task {i}', alpha=0.7)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('KL Divergence')
axes[1].set_title('Minor Tasks (first 5)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Comparison Across Experiments

Compare KL divergence for multiple experiments.

In [ ]:
# Example: Compare multiple experiments
exp_names = [
    # Add your experiment names here
    # "experiment_1",
    # "experiment_2",
]

if len(exp_names) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    for exp_name in exp_names:
        results_exp = compute_kl_divergence_vs_bayes(
            exp_name=exp_name,
            n_minor=N_MINOR,
            batch_size=BATCH_SIZE,
            device=DEVICE,
        )
        
        axes[0].plot(results_exp['positions'], results_exp['kl_major_mean'],
                    label=exp_name, alpha=0.7)
        axes[1].plot(results_exp['positions'], results_exp['kl_minor_mean'],
                    label=exp_name, alpha=0.7)
    
    axes[0].set_xlabel('Position')
    axes[0].set_ylabel('KL Divergence')
    axes[0].set_title('Major Tasks Comparison')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].set_xlabel('Position')
    axes[1].set_ylabel('KL Divergence')
    axes[1].set_title('Minor Tasks Comparison')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Add experiment names to exp_names list to compare multiple experiments")